# Lab 2 — Where Vector Search Breaks (and Where BM25 Saves You)

**Thesis:** Vector search is powerful but not universal. Exact tokens — specific version numbers, config keys, error codes — are compressed by embeddings into generic clusters and then *lost*. BM25 (keyword search) has the opposite failure mode: zero lexical overlap means zero score, even when the answer is obvious.

Understanding *when* each approach fails is the prerequisite to building a hybrid retriever that wins on both.

## What you'll learn
- 3 real query types where semantic search returns the wrong answer
- 1 real query type where BM25 returns zero useful results
- How to read tf/idf score explanations to understand *why* a document ranked where it did
- The core tension table: when to use which approach

## Before you start
- **In Instruqt:** `ES_ENDPOINT` and `ES_API_KEY` are pre-configured — just run the cells.
- **Re-running from the repo:** `export ES_ENDPOINT=https://...` and `export ES_API_KEY=...`

In [ ]:
# --- Workshop helpers (inline — same block across all 4 notebooks) ---

import os, json, time
import requests
from elasticsearch import Elasticsearch

INDEX = "aiewf-workshop-docs"

ES_ENDPOINT = os.environ.get("ES_ENDPOINT")
ES_API_KEY  = os.environ.get("ES_API_KEY")
if not ES_ENDPOINT or not ES_API_KEY:
    raise ValueError(
        "Set ES_ENDPOINT and ES_API_KEY.\n"
        "  In Instruqt: pre-configured in the sandbox.\n"
        "  Re-running the repo: export ES_ENDPOINT=https://...  export ES_API_KEY=..."
    )

es = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY, request_timeout=60)

def show_hits(resp, fields=("id", "title", "trap_type"), score=True):
    """Pretty-print search hits as a ranked table."""
    hits = resp["hits"]["hits"]
    if not hits:
        print("  (no hits)"); return
    for rank, h in enumerate(hits, 1):
        src = h.get("_source", {})
        cols = "  ".join(str(src.get(f, "")) for f in fields)
        s = f"  {h['_score']:.4f}" if score and h.get("_score") is not None else ""
        print(f"  #{rank:<2}{s}  {cols}")

def r_semantic(query):
    return {"standard": {"query": {"semantic": {"field": "body_semantic", "query": query}}}}

def r_bm25(query):
    return {"standard": {"query": {"multi_match": {
        "query": query, "fields": ["title^3", "body"], "type": "best_fields"}}}}

def r_rrf(query, rank_constant=60, rank_window_size=100):
    return {"rrf": {"retrievers": [r_bm25(query), r_semantic(query)],
                    "rank_constant": rank_constant, "rank_window_size": rank_window_size}}

def search(retriever, size=5, source=("id","title","trap_type","version_tags")):
    return es.search(index=INDEX, retriever=retriever, size=size, source=list(source))

# Lab 2 extra: side-by-side comparison helper
def compare(query, size=5):
    """Run the same query through semantic and BM25 side by side."""
    print(f"QUERY: {query!r}\n")
    print("SEMANTIC (vector):")
    show_hits(search(r_semantic(query), size))
    print("\nBM25 (lexical):")
    show_hits(search(r_bm25(query), size))

print("✓ Helpers loaded")

In [ ]:
# Confirm connection
info = es.info()
count = es.count(index=INDEX)["count"]
print(f"Connected to ES {info['version']['number']} | {count} docs in '{INDEX}'")

## Failure mode 1: Exact tokens — vector can't find "137"

Our corpus has a document (`doc-007`) about JVM heap sizing — it covers **OOMKilled / exit code 137**, which is how Kubernetes signals an out-of-memory kill.

The query is: `"exit code 137"`

**What we expect:**
- BM25: `doc-007` should be #1. The doc contains the exact tokens "exit", "code", "137".
- Semantic: `doc-007` might *not* be #1. The embedding model compresses "137" into a region of vector space shared with other numeric codes — it loses the specificity of that exact number.

Watch the semantic column:

In [ ]:
compare("exit code 137")

## Why does semantic miss "exit code 137"?

**The aha moment:**

When the Jina v5 model embeds the query "exit code 137", it produces a vector that captures the *general concept* of a process exit or error code. The number `137` is not treated as a meaningful discriminator — it's compressed into the same embedding neighborhood as "exit code 1", "exit code 2", "crash code", "process failure code", etc.

The document about `OOMKilled / exit 137` is semantically similar to *many* error-related documents. It's not uniquely close to the query in vector space — other error docs have nearly the same distance.

**Rule:** If your users will search with exact identifiers — version numbers, config keys, error codes, usernames, CVE IDs — vector search will blur them. BM25 won't.

---

Let's confirm with two more exact-token queries:

In [ ]:
# Query 2: specific version number — semantic blurs 8.15 vs 8.18 vs 9.x
# We want doc-057 (8.18 release notes) at #1.
compare("8.18 breaking changes")

# Note: watch the version_tags column — did semantic return docs tagged 8.15, 9.0, or 8.18?

In [ ]:
# Query 3: exact configuration key — semantic compresses to generic "security config" cluster
# We want doc-008 (shard allocation settings) — it contains this exact setting name.
compare("xpack.security.authc.realms configuration")

## Failure mode 2: Paraphrase — BM25 can't find "can't log in"

Now the reverse. Our corpus has `doc-001` about SAML authentication problems. The **title** is something like "Configure SAML authentication" and the body discusses `NameIDFormat`, `Issuer`, `assertion consumer service`, `AttributeStatements` — **zero uses of the word "login"**.

A real user types: `"user can't log in"`

**What we expect:**
- Semantic: finds `doc-001` because the *meaning* of "user can't log in" is tightly related to authentication failure / SAML config issues.
- BM25: `doc-001` scores **zero** because there is zero token overlap. BM25 cannot rank what it cannot find.

Watch:

In [ ]:
compare("user can't log in")

## Why does BM25 miss the SAML doc?

`doc-001` contains `authentication`, `identity provider`, `SAML`, `assertion` — every relevant concept *except* the word "login". BM25's core formula is:

```
score(query, doc) = Σ  IDF(term) × TF(term, doc) × length_norm
                  terms in query
```

If a query term doesn't appear in the document at all, its `TF` is 0. So `doc-001`'s BM25 score for `"user can't log in"` is:

- "user" → maybe 0 occurrences in body_semantic? Zero contribution.
- "can't" → stop word, ignored.
- "log" → maybe appears once in a log file reference? Tiny contribution.
- "in" → stop word, ignored.

**Result: effectively zero score.** The most relevant document in the corpus is invisible to BM25.

---

**Rule:** If users describe *what's wrong* rather than the exact vocabulary in the document, BM25 fails. Semantic search handles this gracefully.

## Read the score: how did the engine decide?

For BM25, we can ask Elasticsearch to **explain** exactly how it scored each document: which terms matched, what tf/idf values were used, and why a document ranked where it did.

We'll use `explain=True` on a BM25 query for `"exit code 137"` — this is the query where BM25 wins, so the explanation will show you the tf/idf signal that vector search couldn't replicate.

In [ ]:
# BM25 explanation for the top hit on "exit code 137"
resp = es.search(
    index=INDEX,
    retriever=r_bm25("exit code 137"),
    explain=True,
    size=2,
    source=["id", "title"]
)

for hit in resp["hits"]["hits"]:
    src = hit["_source"]
    print(f"Doc: {src['id']} — {src['title']}")
    print(f"Score: {hit['_score']:.4f}")
    expl = hit.get("_explanation", {})
    # Print the top-level explanation description
    print(f"Explanation: {expl.get('description','')[:200]}")
    # Show top-level sub-explanations (term scores)
    for sub in expl.get("details", [])[:3]:
        print(f"  └─ {sub.get('description','')[:150]}")
    print()

## The semantic score is different in nature

We're intentionally **not** running `explain=True` on a semantic query here. Here's why:

A semantic `explain` tree is huge and opaque — it produces per-chunk cosine similarity scores for every chunk of every document, often hundreds of nested numbers. On stage this creates a wall of JSON that teaches nothing.

What you need to know about the semantic score:

- It's derived from **cosine similarity** between the query vector and each document chunk vector
- Values range roughly from 0 (completely unrelated) to 1 (identical)
- The document's score is the **max similarity across all its chunks** (so long docs don't get penalized)
- Unlike BM25, it is **not** sensitive to document length or term frequency — it's purely about vector proximity

The practical implication: BM25 scores and semantic scores are on **different scales** with different distributions. You can't just add them — which is exactly why hybrid fusion needs normalization. That's Lab 3.

## The core tension — summary table

| Query type | Example | Semantic wins? | BM25 wins? |
|---|---|---|---|
| Paraphrase / meaning | `"user can't log in"` | ✅ yes | ❌ zero overlap |
| Exact token | `"exit code 137"` | ❌ blurred | ✅ exact match |
| Version-specific | `"8.18 breaking changes"` | ❌ blurs versions | ✅ exact number |
| Config key | `"xpack.security.authc.realms"` | ❌ generic cluster | ✅ exact token |
| Natural language | `"securing cluster traffic"` | ✅ concept match | ⚠ depends on vocabulary |

**The conclusion:** neither approach wins universally. A production search system needs both — and a way to fuse their rankings. That's what RRF and linear combination do.

---
*Continue in Dev Console → Lab 3 assignment, or open `lab3-hybrid-search.ipynb`*